## 1. Extracting Data from Azure Blob Storage

In [1]:
# Loading data into pyspark from azure
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession

# get storage account key from .env
load_dotenv()
key = os.getenv("storage_account_key")

# create spark session
storage_account = "data479projectg5"
container = "raw-data"
spark = SparkSession.builder.appName("GSOD_DataExploration")\
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-azure:3.3.4")\
    .config(f"fs.azure.account.key.{storage_account}.blob.core.windows.net", key)\
    .getOrCreate()

# get csv paths from azure and load them all in as a dataframe in spark
years = [1999, 2000, 2001, 2009, 2010, 2011, 2019, 2020, 2021] # specify years
csv_paths = [f"wasbs://{container}@{storage_account}.blob.core.windows.net/{year}/*" for year in years] # get csv paths from azure for each year
df = spark.read.csv(csv_paths, header=True, inferSchema=True) # load csvs into dataframe 

print("data loaded successfully") # prints if no errors occur during the loading process

data loaded successfully


In [2]:
# Get data schema/data types 
df.printSchema()

root
 |-- STATION: string (nullable = true)
 |-- DATE: date (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)
 |-- ELEVATION: double (nullable = true)
 |-- NAME: string (nullable = true)
 |-- TEMP: double (nullable = true)
 |-- TEMP_ATTRIBUTES: double (nullable = true)
 |-- DEWP: double (nullable = true)
 |-- DEWP_ATTRIBUTES: double (nullable = true)
 |-- SLP: double (nullable = true)
 |-- SLP_ATTRIBUTES: double (nullable = true)
 |-- STP: double (nullable = true)
 |-- STP_ATTRIBUTES: double (nullable = true)
 |-- VISIB: double (nullable = true)
 |-- VISIB_ATTRIBUTES: double (nullable = true)
 |-- WDSP: double (nullable = true)
 |-- WDSP_ATTRIBUTES: double (nullable = true)
 |-- MXSPD: double (nullable = true)
 |-- GUST: double (nullable = true)
 |-- MAX: double (nullable = true)
 |-- MAX_ATTRIBUTES: string (nullable = true)
 |-- MIN: double (nullable = true)
 |-- MIN_ATTRIBUTES: string (nullable = true)
 |-- PRCP: double (nullable = tru

---

## 2. Data Pre-Processing

Before proceeding with analysis we will first check and see if there are any records which are not part of the list of 40 stations specified in the data set (e.g. null records).

In [3]:
from pyspark.sql.functions import col

# get list of all 40 stations included in the dataset
stations = []
with open("stations.txt") as f:
    for line in f:
        id = line.strip().split('- ')[1]
        stations.append(id)

# filter to any stations which are not in the list
df.filter(~col("STATION").isin(stations)).show()

+--------------------+----+--------+---------+---------+----+----+---------------+----+---------------+----+--------------+----+--------------+-----+----------------+----+---------------+-----+----+----+--------------+----+--------------+----+---------------+----+------+
|             STATION|DATE|LATITUDE|LONGITUDE|ELEVATION|NAME|TEMP|TEMP_ATTRIBUTES|DEWP|DEWP_ATTRIBUTES| SLP|SLP_ATTRIBUTES| STP|STP_ATTRIBUTES|VISIB|VISIB_ATTRIBUTES|WDSP|WDSP_ATTRIBUTES|MXSPD|GUST| MAX|MAX_ATTRIBUTES| MIN|MIN_ATTRIBUTES|PRCP|PRCP_ATTRIBUTES|SNDP|FRSHTT|
+--------------------+----+--------+---------+---------+----+----+---------------+----+---------------+----+--------------+----+--------------+-----+----------------+----+---------------+-----+----+----+--------------+----+--------------+----+---------------+----+------+
|<Error><Code>NoSu...|NULL|    NULL|     NULL|     NULL|NULL|NULL|           NULL|NULL|           NULL|NULL|          NULL|NULL|          NULL| NULL|            NULL|NULL|           NU

We can see that there are a few null records that occurred when loading in the dataset. We can remove these entries from the dataset as they do not provide any contribution or value to our analysis or the dataset.

In [4]:
# Filter data to just records with a station ID included in the list of 40 stations
df = df.filter(col("STATION").isin(stations))

---

## 3. Exploratory Data Analysis

In [5]:
# Count the total number of records in the dataset (entries from all 40 stations across all 9 years)
print(f"Total Number of Records: {df.count()}")

Total Number of Records: 127839


In [6]:
# Count the number of unique weather stations identified in the dataset
print(f"Number of Weather Stations: {df.select("STATION").distinct().count()}")

Number of Weather Stations: 40


In [7]:
# Count the number of years covered in this dataset
from pyspark.sql.functions import year, col

years = df.select(year(col("DATE")).alias("YEAR"))\
                          .distinct()\
                          .orderBy("YEAR")

years.show()
print(f"Years Covered: {years.count()}")

+----+
|YEAR|
+----+
|1999|
|2000|
|2001|
|2009|
|2010|
|2011|
|2019|
|2020|
|2021|
+----+

Years Covered: 9


We will now identify any missing or invalid values (marked as `9999.9` in the dataset) in the temperature fields of the dataset.

We have `TEMP`, `MAX`, and `MIN` as the main temperature fields in the dataset where:
- `TEMP` = average temperature that day
- `MAX` = highest temperature recorded that day
- `MIN` = lowest temperature recorded that day

In [8]:
# filter to check for any missing or invalid values (denoted with 9999.9) in any of the temperature fields
missing = df.filter(
        (col("TEMP") == 9999.9) | 
        (col("MAX") == 9999.9) | 
        (col("MIN") == 9999.9))\
    .select("STATION", "DATE", "NAME", "TEMP", "MIN", "MAX")

missing.show()
print(f"Number of Records with Missing Data: {missing.count()}")

+-----------+----------+--------------------+-----+------+------+
|    STATION|      DATE|                NAME| TEMP|   MIN|   MAX|
+-----------+----------+--------------------+-----+------+------+
|71866099999|2000-09-27|SASKATOON J G DIE...| 46.9|  33.8|9999.9|
|71866099999|2020-02-12|SASKATOON J G DIE...| -7.8| -23.8|9999.9|
|72446003947|2020-10-23|KANSAS CITY INTER...| 44.4|  37.0|9999.9|
|72446003947|2011-02-21|KANSAS CITY INTER...| 33.9|  24.1|9999.9|
|72446003947|2011-04-04|KANSAS CITY INTER...| 49.4|  42.1|9999.9|
|83827099999|2020-01-11|CATARATAS INTERNA...| 76.2|9999.9|  91.0|
|83827099999|2020-02-25|CATARATAS INTERNA...| 78.4|9999.9|  92.1|
|83827099999|2020-12-20|CATARATAS INTERNA...| 72.6|9999.9|  95.0|
|71877099999|2009-05-18|CALGARY INTERNATI...| 41.8|  30.2|9999.9|
|71877099999|2019-01-19|CALGARY INTERNATI...|  7.6|  -0.4|9999.9|
|71877099999|2019-10-08|CALGARY INTERNATI...| 33.2|  23.0|9999.9|
|71852099999|2020-02-12|WINNIPEG INTERNAT...| -5.0| -22.0|9999.9|
|718520999

There are 10 stations which contain invalid minimum (`MIN`) or maximum (`MAX`) temperature values in this dataset.

---